# Project 14 — Phát hiện xâm nhập mạng với công cụ có sẵn

**Công cụ:** Google Colab, pandas, scikit-learn, SciPy và matplotlib. Chạy các ô theo thứ tự từ trên xuống; thuật toán đã được thư viện cài đặt.

**Thực nghiệm:** K-Means + Euclidean, K-Means + Mahalanobis (đổi cách chấm điểm, không đổi hàm mục tiêu K-Means), DBSCAN + khoảng cách tới điểm lõi, Random Forest.

**Thiết lập trung thực:** K-Means/DBSCAN học từ dữ liệu hỗn hợp không dùng nhãn; nhưng chọn tham số và ngưỡng có dùng nhãn validation. Đây là phân cụm không giám sát với hiệu chỉnh dựa trên nhãn, không phải pipeline hoàn toàn không giám sát. Không có bảo đảm mô hình không giám sát tốt hơn mô hình phân loại.

**Dữ liệu thật chưa được chạy trong tệp mẫu này.** Các ô đầu ra để trống. Luồng tính toán đã được kiểm tra bằng dữ liệu tổng hợp để kiểm tra việc thực thi, không phải để kết luận chất lượng trên NSL-KDD.


## 1. Mở trên Colab

1. Vào https://colab.research.google.com/ → **File → Upload notebook**, chọn tệp này.
2. Chọn runtime Python, bộ tăng tốc **None/CPU**. Các thuật toán scikit-learn ở đây không cần GPU.
3. Lưu bản sao notebook vào Drive. Dữ liệu và kết quả trong `/content` sẽ mất khi runtime bị xóa.
4. Chạy từng ô bằng nút ▶ hoặc Shift+Enter. Đọc kết quả rồi mới chạy phần tiếp theo.


In [ ]:
%pip -q install "scikit-learn>=1.2,<2" pandas numpy scipy matplotlib seaborn joblib


In [ ]:
from pathlib import Path
import json, time, platform, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sklearn
from scipy.spatial.distance import cdist
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.covariance import LedoitWolf
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (precision_score, recall_score, f1_score, confusion_matrix,
    roc_auc_score, average_precision_score, ConfusionMatrixDisplay,
    PrecisionRecallDisplay, classification_report)

SEED = 42
# Chạy thử với cùng tối đa 10.000 bản ghi phát triển cho TẤT CẢ mô hình.
# Đây là mẫu hỗn hợp ngẫu nhiên, không chọn riêng normal, không dùng test.
# Đặt None để dùng toàn bộ train chính thức; DBSCAN có thể tốn nhiều RAM/thời gian.
MAX_DEVELOPMENT_ROWS = 10_000
VALIDATION_SIZE = 0.2
MAX_VAL_FPR = 0.05  # Giới hạn FPR trên validation; không bảo đảm FPR test <= 5%.
K_VALUES = [8, 16, 32]
MIN_SAMPLES = 10
EPS_QUANTILES = [0.60, 0.80, 0.95]

ROOT = Path('/content/project14') if Path('/content').exists() else Path.cwd() / 'project14_run'
DATA = ROOT / 'data'
OUT = ROOT / 'outputs'
for folder in [DATA, OUT / 'figures', OUT / 'models']:
    folder.mkdir(parents=True, exist_ok=True)
print('Thư mục:', ROOT)
print('Python:', platform.python_version(), '| scikit-learn:', sklearn.__version__)


## 2. Tải và đưa dataset lên Colab

1. Mở https://www.kaggle.com/datasets/hassan06/nslkdd và tải dataset (đăng nhập nếu được yêu cầu).
2. Giải nén trên máy. Chọn đúng **KDDTrain+.txt** và **KDDTest+.txt**, không dùng bản 20Percent hay Test-21 cho thí nghiệm chính.
3. Chạy ô dưới và chọn đồng thời hai tệp TXT. Không tải nhầm tệp ARFF.

Tệp TXT chuẩn không có header; thường có 43 cột: 41 đặc trưng, nhãn, độ khó. Bộ đọc cũng chấp nhận bản 42 cột không có độ khó. Nếu dùng ngoài Colab, chép hai tệp vào thư mục `DATA` được in ở trên.


In [ ]:
try:
    from google.colab import files
except ImportError:
    print('Ngoài Colab: chép hai tệp TXT vào', DATA)
else:
    uploaded = files.upload()
    for name, payload in uploaded.items():
        if Path(name).name in {'KDDTrain+.txt', 'KDDTest+.txt'}:
            (DATA / Path(name).name).write_bytes(payload)
    del uploaded

for filename in ['KDDTrain+.txt', 'KDDTest+.txt']:
    if not (DATA / filename).exists():
        raise FileNotFoundError(f'Chưa có {filename} trong {DATA}')


In [ ]:
FEATURES = [
 'duration','protocol_type','service','flag','src_bytes','dst_bytes','land',
 'wrong_fragment','urgent','hot','num_failed_logins','logged_in','num_compromised',
 'root_shell','su_attempted','num_root','num_file_creations','num_shells',
 'num_access_files','num_outbound_cmds','is_host_login','is_guest_login',
 'count','srv_count','serror_rate','srv_serror_rate','rerror_rate','srv_rerror_rate',
 'same_srv_rate','diff_srv_rate','srv_diff_host_rate','dst_host_count',
 'dst_host_srv_count','dst_host_same_srv_rate','dst_host_diff_srv_rate',
 'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate','dst_host_serror_rate',
 'dst_host_srv_serror_rate','dst_host_rerror_rate','dst_host_srv_rerror_rate'
]
CAT = ['protocol_type', 'service', 'flag']
LOG = ['duration', 'src_bytes', 'dst_bytes']
BINARY = ['land', 'logged_in', 'root_shell', 'is_host_login', 'is_guest_login']
NUM = [c for c in FEATURES if c not in CAT + LOG + BINARY]
assert len(FEATURES) == 41 and len(set(FEATURES)) == 41

def read_nsl(path):
    frame = pd.read_csv(path, header=None)
    if frame.shape[1] not in [42, 43]:
        raise ValueError(f'{path.name}: cần 42/43 cột, thấy {frame.shape[1]}. Kiểm tra định dạng.')
    frame.columns = FEATURES + ['attack_name'] + (['difficulty'] if frame.shape[1] == 43 else [])
    for c in CAT:
        frame[c] = frame[c].astype('string').str.strip().replace('', pd.NA)
        frame[c] = frame[c].astype(object).where(frame[c].notna(), np.nan)
    for c in LOG + BINARY + NUM:
        frame[c] = pd.to_numeric(frame[c], errors='raise').replace([np.inf, -np.inf], np.nan)
    if (frame[LOG] < 0).any().any():
        raise ValueError('Có giá trị âm ở biến cần log1p. Hãy kiểm tra dữ liệu.')
    if frame['attack_name'].isna().any():
        raise ValueError('Nhãn bị thiếu.')
    frame['attack_name'] = frame['attack_name'].astype(str).str.strip().str.lower().str.rstrip('.')
    frame['target'] = (frame['attack_name'] != 'normal').astype('int8')
    return frame

train_original = read_nsl(DATA / 'KDDTrain+.txt')
test_original = read_nsl(DATA / 'KDDTest+.txt')
print('Train gốc:', train_original.shape, '| Test gốc:', test_original.shape)
display(train_original.head())


## 3. EDA — Khảo sát dữ liệu phát triển

Chỉ khảo sát train ở bước này; giữ test cho đánh giá cuối. Cột `attack_name`, `target`, `difficulty` không phải đặc trưng đầu vào. Đừng xóa các giá trị lớn chỉ vì trông giống outlier: chúng có thể là tấn công cần phát hiện.

Ghi vào báo cáo: số bản ghi, số đặc trưng, thiếu dữ liệu, cột hằng, tỷ lệ attack/normal, các loại tấn công ít mẫu và độ lệch của byte counts.


In [ ]:
print('Nhãn nhị phân:')
display(train_original['target'].value_counts().rename(index={0:'normal', 1:'attack'}))
print('Giá trị thiếu (chỉ cột có thiếu):')
missing = train_original[FEATURES].isna().sum()
display(missing[missing > 0])
print('Cột hằng:', [c for c in FEATURES if train_original[c].nunique(dropna=False) <= 1])
display(train_original[LOG + ['count', 'srv_count']].describe())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
train_original['attack_name'].value_counts().head(12).plot.bar(ax=axes[0])
axes[0].set_title('Các nhãn phổ biến trong train gốc')
np.log1p(train_original['src_bytes']).hist(bins=60, ax=axes[1])
axes[1].set_title('Phân bố log1p(src_bytes)')
fig.tight_layout()
fig.savefig(OUT / 'figures' / '01_eda.png', dpi=160)
plt.show()


## 4. Chia dữ liệu — không trộn train với test

- Chọn ngẫu nhiên tối đa `MAX_DEVELOPMENT_ROWS` từ train gốc để chạy vừa sức; tất cả mô hình dùng cùng ngân sách dữ liệu.
- Chia mẫu này thành **fit/validation = 80/20**, phân tầng theo normal/attack. Nhãn chỉ giúp phân chia cân đối; không truyền nhãn vào K-Means/DBSCAN.
- Test giữ nguyên. Không fit scaler/PCA trên test, không chọn ngưỡng bằng test.
- Trong báo cáo, phân biệt **novel chính thức** (tên không có trong toàn bộ KDDTrain+) và **chưa thấy trong dữ liệu phát triển đã chọn**. Tập thứ hai có thể lớn hơn do lấy mẫu; không đánh đồng chúng.


In [ ]:
n_dev = len(train_original) if MAX_DEVELOPMENT_ROWS is None else min(MAX_DEVELOPMENT_ROWS, len(train_original))
development = train_original.sample(n=n_dev, random_state=SEED).copy()
fit_df, val_df = train_test_split(development, test_size=VALIDATION_SIZE,
    stratify=development['target'], random_state=SEED)
X_fit_raw, y_fit = fit_df[FEATURES], fit_df['target'].to_numpy()
X_val_raw, y_val = val_df[FEATURES], val_df['target'].to_numpy()

official_attack_names = set(train_original.loc[train_original.target == 1, 'attack_name'])
development_attack_names = set(development.loc[development.target == 1, 'attack_name'])
fit_attack_names = set(fit_df.loc[fit_df.target == 1, 'attack_name'])
print('Fit:', len(fit_df), '| Validation:', len(val_df), '| Test giữ nguyên:', len(test_original))
print('Số tên tấn công: train gốc / development / fit:',
      len(official_attack_names), len(development_attack_names), len(fit_attack_names))


## 5. Tiền xử lý bằng công cụ có sẵn

- Ba biến chữ → `OneHotEncoder(handle_unknown='ignore')`.
- Duration/byte counts → impute median → log1p → StandardScaler.
- Biến số khác → impute median → StandardScaler; các cờ nhị phân giữ thang 0/1.
- Bỏ cột hằng, sau đó PCA giữ 95% phương sai để giảm số chiều và chi phí DBSCAN.
- Tất cả mô hình dùng cùng biểu diễn sau PCA. Đây là thiết lập so sánh thống nhất, không khẳng định là biểu diễn tối ưu cho Random Forest.

**Lưu ý:** giá trị phân loại chưa thấy được mã hóa thành toàn số 0 trong nhóm one-hot đó; không có nghĩa nó chắc chắn là tấn công. Các biến đổi chỉ học từ fit.


In [ ]:
log_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('log1p', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
    ('scale', StandardScaler())])
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scale', StandardScaler())])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=np.float32))])
columns = ColumnTransformer([
    ('log', log_pipeline, LOG), ('num', num_pipeline, NUM),
    ('binary', SimpleImputer(strategy='most_frequent'), BINARY),
    ('cat', cat_pipeline, CAT)], sparse_threshold=0)
preprocessor = Pipeline([
    ('columns', columns), ('variance', VarianceThreshold()),
    ('pca', PCA(n_components=0.95, svd_solver='full'))])

t0 = time.perf_counter()
X_fit = preprocessor.fit_transform(X_fit_raw).astype(np.float32)
X_val = preprocessor.transform(X_val_raw).astype(np.float32)
preprocess_fit_s = time.perf_counter() - t0
print('Ma trận fit:', X_fit.shape, '| Validation:', X_val.shape)
print('Phương sai PCA giữ lại:', preprocessor['pca'].explained_variance_ratio_.sum())
print('Thời gian fit + biến đổi validation:', round(preprocess_fit_s, 2), 'giây')
assert np.isfinite(X_fit).all() and np.isfinite(X_val).all()


## 6. Quy tắc chọn ngưỡng và hàm chấm điểm

Quy ước **1 = attack**, điểm càng cao càng đáng ngờ. Với mỗi mô hình, thử các ngưỡng trên validation, ưu tiên F1 cao nhất trong số các ngưỡng có FPR validation ≤ 5%. Ngưỡng và cấu hình chốt trước khi chạy test.

Đây là **hiệu chỉnh có sử dụng nhãn validation**. Giới hạn 5% là mục tiêu thí nghiệm trên validation, không phải bảo đảm ngoài thực tế. Nếu bỏ giới hạn này trong lần thử khác, phải ghi rõ và vẫn không điều chỉnh dựa vào test.

Đối với DBSCAN, điểm là khoảng cách Euclidean tới điểm lõi gần nhất trong fit. Quy tắc gốc `distance > eps` được báo cáo riêng; biến thể có hiệu chỉnh ngưỡng phải được ghi tên rõ ràng.


In [ ]:
def binary_metrics(y, pred, scores=None):
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    result = dict(precision=precision_score(y, pred, zero_division=0),
        recall=recall_score(y, pred, zero_division=0),
        f1=f1_score(y, pred, zero_division=0), fpr=fp / max(tn + fp, 1),
        tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))
    if scores is not None and len(np.unique(y)) == 2:
        result.update(roc_auc=roc_auc_score(y, scores),
                      average_precision=average_precision_score(y, scores))
    return result

def choose_threshold(y, scores):
    # Candidate quantiles + ngưỡng không cảnh báo điểm nào.
    thresholds = np.unique(np.r_[np.quantile(scores, np.linspace(0, 1, 151)),
        np.nextafter(float(np.max(scores)), np.inf)])
    options = []
    for threshold in thresholds:
        m = binary_metrics(y, scores >= threshold)
        if m['fpr'] <= MAX_VAL_FPR:
            options.append((m['f1'], m['recall'], -m['fpr'], float(threshold), m))
    winner = max(options, key=lambda a: a[:3])
    return winner[3], winner[4]

def anomaly_scores(spec, X):
    kind = spec['kind']
    if kind == 'kmeans_euclidean':
        score = spec['model'].transform(X).min(axis=1)
    elif kind == 'kmeans_mahalanobis':
        # Precision = L @ L.T, nên Euclidean trong X @ L bằng Mahalanobis.
        L = spec['precision_cholesky']
        centers = spec['model'].cluster_centers_ @ L
        score = cdist(X @ L, centers, metric='euclidean').min(axis=1)
    elif kind == 'dbscan_core_distance':
        score = spec['nearest_core'].kneighbors(X, return_distance=True)[0][:, 0]
    elif kind == 'random_forest':
        index = list(spec['model'].classes_).index(1)
        score = spec['model'].predict_proba(X)[:, index]
    else:
        raise ValueError(kind)
    return np.asarray(score, dtype=np.float64)

candidates, validation_rows = {}, []
def register(name, spec):
    val_scores = anomaly_scores(spec, X_val)
    threshold, m = choose_threshold(y_val, val_scores)
    spec['threshold'] = threshold
    spec['validation_f1'] = m['f1']
    candidates[name] = spec
    validation_rows.append(dict(name=name, threshold=threshold,
        fit_s=spec['fit_s'], **m))
    print(name, '| F1 val:', round(m['f1'], 4), '| FPR val:', round(m['fpr'], 4))


## 7. K-Means + Euclidean và Mahalanobis

Thử K = 8, 16, 32. Euclidean là khoảng cách đến tâm gần nhất. Mahalanobis ở đây dùng **hiệp phương sai toàn cục được điều chuẩn bởi LedoitWolf**, rồi tính khoảng cách đến các tâm đã học bằng K-Means Euclidean. Đây là thay đổi cách chấm điểm, không phải một thuật toán K-Means với mục tiêu Mahalanobis.

Các cụm có thể chứa cả normal và attack. Một đợt tấn công dày đặc có thể nằm gần tâm nên bị bỏ sót; đó là hạn chế cần phân tích.


In [ ]:
t0 = time.perf_counter()
covariance = LedoitWolf().fit(X_fit)
precision_cholesky = np.linalg.cholesky(covariance.precision_)
cov_fit_s = time.perf_counter() - t0

for k in K_VALUES:
    if k >= len(X_fit):
        continue
    t0 = time.perf_counter()
    model = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X_fit)
    fit_s = time.perf_counter() - t0
    register(f'KMeans-Euclidean-K{k}', dict(kind='kmeans_euclidean',
        model=model, fit_s=fit_s, k=k))
    register(f'KMeans-Mahalanobis-K{k}', dict(kind='kmeans_mahalanobis',
        model=model, precision_cholesky=precision_cholesky,
        fit_s=fit_s + cov_fit_s, k=k))


## 8. DBSCAN + quy tắc cho dữ liệu mới

DBSCAN không có `predict` chuẩn cho mẫu mới trong scikit-learn. Notebook giữ các điểm lõi trong fit, xây dựng tìm láng giềng gần nhất rồi dùng khoảng cách tới điểm lõi làm score. Không chạy `fit_predict` trên test.

Ước lượng eps bằng các phân vị khoảng cách tới láng giềng thứ `min_samples`, sau đó chọn cấu hình bằng validation. Nếu không có điểm lõi thì cấu hình đó bị bỏ qua.

**Tài nguyên:** DBSCAN có thể tốn RAM lớn. Khi bị ngắt vì RAM, giảm `MAX_DEVELOPMENT_ROWS` còn 5.000 và chạy lại từ bước 4 cho tất cả mô hình, không giảm riêng dữ liệu DBSCAN mà quên công khai. Giảm dữ liệu có thể làm mất lớp hiếm; ghi rõ số mẫu và seed trong báo cáo.


In [ ]:
if len(X_fit) < MIN_SAMPLES:
    raise ValueError('Tập fit quá nhỏ cho MIN_SAMPLES.')
nn_density = NearestNeighbors(n_neighbors=MIN_SAMPLES, n_jobs=-1).fit(X_fit)
k_dist = nn_density.kneighbors(X_fit)[0][:, -1]
eps_values = np.unique(np.maximum(np.quantile(k_dist, EPS_QUANTILES), 1e-6))
plt.figure(figsize=(7, 3))
plt.plot(np.sort(k_dist))
plt.title('Khoảng cách láng giềng đã sắp xếp — hỗ trợ chọn eps')
plt.xlabel('Bản ghi fit (đã sắp xếp)')
plt.ylabel('Khoảng cách')
plt.tight_layout()
plt.savefig(OUT / 'figures' / '02_k_distance.png', dpi=160)
plt.show()

for i, eps in enumerate(eps_values):
    t0 = time.perf_counter()
    model = DBSCAN(eps=float(eps), min_samples=MIN_SAMPLES,
        metric='euclidean', n_jobs=-1).fit(X_fit)
    if len(model.components_) == 0:
        print('Bỏ eps', eps, ': không có điểm lõi.')
        continue
    nearest_core = NearestNeighbors(n_neighbors=1, n_jobs=-1).fit(model.components_)
    fit_s = time.perf_counter() - t0
    # Ghi rõ noise của dữ liệu fit không phải kết quả kiểm thử.
    print('eps:', round(float(eps), 4), '| Tỷ lệ noise trên fit:', round(np.mean(model.labels_ == -1), 4))
    register(f'DBSCAN-core-distance-{i}', dict(kind='dbscan_core_distance',
        model=model, nearest_core=nearest_core, eps=float(eps), fit_s=fit_s))


## 9. Random Forest — mô hình có giám sát để so sánh

Sử dụng cùng X_fit, X_val nhưng có thêm y_fit khi huấn luyện. `class_weight='balanced'` là lựa chọn ban đầu; không áp dụng SMOTE vào các nhánh phân cụm trong thí nghiệm này.

Thử hai độ sâu; cùng chọn ngưỡng bằng validation và cùng giới hạn FPR như các mô hình khác.


In [ ]:
for depth in [16, None]:
    t0 = time.perf_counter()
    model = RandomForestClassifier(n_estimators=150, max_depth=depth,
        min_samples_leaf=2, class_weight='balanced', n_jobs=-1,
        random_state=SEED).fit(X_fit, y_fit)
    fit_s = time.perf_counter() - t0
    register(f'RandomForest-depth-{depth}', dict(kind='random_forest',
        model=model, fit_s=fit_s, depth=depth))

validation_table = pd.DataFrame(validation_rows).sort_values('f1', ascending=False)
display(validation_table)
validation_table.to_csv(OUT / 'validation_candidates.csv', index=False)


## 10. Chốt cấu hình trước khi đánh giá test

Chọn một cấu hình tốt nhất cho mỗi họ mô hình bằng F1 validation, sau đó Recall và FPR để phá hòa. Giữ nguyên mô hình đã fit và ngưỡng đã chọn; không refit lên fit+validation vì việc đó có thể thay đổi thang score và cần hiệu chỉnh lại.

Thêm DBSCAN với ngưỡng eps gốc làm kết quả tham khảo, để không nhầm biến thể hiệu chỉnh với quy tắc mật độ gốc.


In [ ]:
selected = {}
for kind in ['kmeans_euclidean', 'kmeans_mahalanobis', 'dbscan_core_distance', 'random_forest']:
    names = [name for name, spec in candidates.items() if spec['kind'] == kind]
    if names:
        rows = validation_table[validation_table.name.isin(names)].sort_values(
            ['f1', 'recall', 'fpr'], ascending=[False, False, True])
        best_name = rows.iloc[0]['name']
        selected[best_name] = candidates[best_name]

db_names = [name for name, spec in selected.items() if spec['kind'] == 'dbscan_core_distance']
if db_names:
    original = selected[db_names[0]]
    reference = dict(original)
    # Dùng >= threshold; nextafter tương đương điều kiện khoảng cách > eps.
    reference['threshold'] = np.nextafter(original['eps'], np.inf)
    selected['DBSCAN-reference-distance>eps'] = reference
print('Cấu hình đã khóa:', list(selected))


## 11. Đánh giá test một lần và tấn công mới

Hai cách gắn cờ được công khai:

1. **official_novel:** tên attack không xuất hiện trong toàn bộ KDDTrain+ — chỉ số chính để so sánh theo split chuẩn.
2. **development_unseen:** tên attack không xuất hiện trong cả fit lẫn validation đã chọn — kiểm tra bổ sung khi chạy lấy mẫu.

`known` trong bảng chính nghĩa là đã có trong KDDTrain+; do lấy mẫu, không bảo đảm mọi tên đó có mẫu trong fit. Bảng xuất kèm cờ `seen_in_fit`, `seen_in_development` để kiểm chứng. Không gọi kết quả này là chứng minh phát hiện mọi zero-day thực tế.


In [ ]:
X_test = preprocessor.transform(test_original[FEATURES]).astype(np.float32)
y_test = test_original['target'].to_numpy()
attack_mask = y_test == 1
official_novel = attack_mask & ~test_original.attack_name.isin(official_attack_names).to_numpy()
development_unseen = attack_mask & ~test_original.attack_name.isin(development_attack_names).to_numpy()
official_known = attack_mask & ~official_novel
print('Test normal:', int((~attack_mask).sum()), '| known attack:', int(official_known.sum()),
      '| official novel attack:', int(official_novel.sum()))
print('Tên novel chính thức:', sorted(set(test_original.loc[official_novel, 'attack_name'])))

def subset_recall(pred, mask):
    return float(np.mean(pred[mask] == 1)) if mask.any() else np.nan

test_rows, predictions, test_scores = [], {}, {}
prediction_table = test_original[['attack_name', 'target']].copy()
prediction_table.insert(0, 'test_row_id', np.arange(len(test_original)))
prediction_table['official_novel'] = official_novel
prediction_table['development_unseen'] = development_unseen
prediction_table['seen_in_fit'] = test_original.attack_name.isin(fit_attack_names).to_numpy()
prediction_table['seen_in_development'] = test_original.attack_name.isin(development_attack_names).to_numpy()

for name, spec in selected.items():
    t0 = time.perf_counter()
    scores = anomaly_scores(spec, X_test)
    pred = (scores >= spec['threshold']).astype('int8')
    score_s = time.perf_counter() - t0
    predictions[name], test_scores[name] = pred, scores
    test_rows.append(dict(model=name, **binary_metrics(y_test, pred, scores),
        recall_known=subset_recall(pred, official_known),
        recall_novel=subset_recall(pred, official_novel),
        recall_development_unseen=subset_recall(pred, development_unseen),
        n_known=int(official_known.sum()), n_novel=int(official_novel.sum()),
        threshold=float(spec['threshold']), fit_s=spec['fit_s'], score_test_s=score_s))
    prediction_table[name + '__score'] = scores
    prediction_table[name + '__prediction'] = pred

test_results = pd.DataFrame(test_rows)
display(test_results[['model','precision','recall','f1','fpr','average_precision',
    'recall_known','recall_novel','fit_s','score_test_s']])
test_results.to_csv(OUT / 'test_results.csv', index=False)
prediction_table.to_csv(OUT / 'test_predictions.csv', index=False)

per_attack_rows = []
for name, pred in predictions.items():
    for attack_name in sorted(test_original.attack_name.unique()):
        mask = (test_original.attack_name == attack_name).to_numpy()
        per_attack_rows.append(dict(model=name, attack_name=attack_name, n=int(mask.sum()),
            is_official_novel=(attack_name != 'normal' and attack_name not in official_attack_names),
            detection_rate=float(pred[mask].mean())))
per_attack = pd.DataFrame(per_attack_rows)
per_attack.to_csv(OUT / 'detection_by_attack.csv', index=False)
display(per_attack.head(15))
print('Với normal, detection_rate là tỷ lệ báo nhầm; với attack, đó là Recall.')
print('fit_s không gồm preprocessing; score_test_s không gồm biến đổi dữ liệu, không phải độ trễ toàn hệ thống.')


## 12. Biểu đồ và phân tích sai số

Confusion Matrix đọc theo thứ tự normal/attack. Average Precision và đường Precision–Recall sử dụng score liên tục, không sử dụng nhãn 0/1 làm score.

Chọn mô hình để minh họa theo validation, không chọn lại bằng điểm test. PCA scatter dùng hai trục đầu của biểu diễn đã học từ fit; chỉ để minh họa, không chứng minh các cụm trong không gian đầy đủ tách biệt.


In [ ]:
ncols = 2
nrows = int(np.ceil(len(predictions) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4 * nrows), squeeze=False)
for ax, (name, pred) in zip(axes.flat, predictions.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, pred,
        display_labels=['normal', 'attack'], ax=ax, colorbar=False)
    ax.set_title(name, fontsize=10)
for ax in list(axes.flat)[len(predictions):]:
    ax.axis('off')
fig.tight_layout()
fig.savefig(OUT / 'figures' / '03_confusion_matrices.png', dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
for name, scores in test_scores.items():
    if name != 'DBSCAN-reference-distance>eps':
        PrecisionRecallDisplay.from_predictions(y_test, scores, name=name, ax=ax)
ax.set_title('Precision–Recall trên test')
fig.tight_layout()
fig.savefig(OUT / 'figures' / '04_precision_recall.png', dpi=160)
plt.show()

test_results.set_index('model')[['f1','recall','fpr','recall_novel']].plot.bar(figsize=(12, 5))
plt.ylim(0, 1)
plt.ylabel('Tỷ lệ')
plt.title('So sánh chỉ số — FPR thấp hơn là tốt hơn')
plt.tight_layout()
plt.savefig(OUT / 'figures' / '05_comparison.png', dpi=160)
plt.show()

primary_name = max([n for n in selected if n != 'DBSCAN-reference-distance>eps'],
    key=lambda n: selected[n]['validation_f1'])
error_mask = predictions[primary_name] != y_test
print('Mô hình minh họa chọn bằng validation:', primary_name)
display(prediction_table.loc[error_mask, ['test_row_id','attack_name','target',
    'official_novel', primary_name+'__score', primary_name+'__prediction']].head(20))
print(classification_report(y_test, predictions[primary_name],
    target_names=['normal','attack'], zero_division=0))

if X_test.shape[1] >= 2:
    rng = np.random.default_rng(SEED)
    ids = rng.choice(len(X_test), size=min(4000, len(X_test)), replace=False)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, labels, title in [(axes[0], y_test, 'Nhãn thật'),
        (axes[1], predictions[primary_name], 'Dự đoán: ' + primary_name)]:
        for value, label, color in [(0, 'normal', 'tab:blue'), (1, 'attack', 'tab:red')]:
            subset = ids[labels[ids] == value]
            ax.scatter(X_test[subset, 0], X_test[subset, 1], s=7, alpha=.4,
                label=label, c=color)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / 'figures' / '06_pca.png', dpi=160)
    plt.show()


## 13. Lưu mô hình, dự đoán thử và tải kết quả

Gói model lưu cả tiền xử lý, tham số, ngưỡng, tên đặc trưng và phiên bản thư viện. Khi dùng lại, giữ môi trường tương thích. `joblib` chỉ nên mở với tệp do nhóm tạo hoặc nguồn tin cậy.

Tệp ZIP bên dưới chứa kết quả, biểu đồ và mô hình; không chứa dataset gốc. Tải ngay để không mất khi runtime Colab kết thúc. Notebook cũng cần được lưu/tải riêng.


In [ ]:
manifest = dict(seed=SEED, max_development_rows=MAX_DEVELOPMENT_ROWS,
    n_train_original=len(train_original), n_fit=len(fit_df), n_validation=len(val_df),
    n_test=len(test_original), max_validation_fpr=MAX_VAL_FPR,
    protocol='mixed-data clustering; label-calibrated validation; fixed official test',
    novelty='official_novel = attack_name absent from entire KDDTrain+',
    sklearn_version=sklearn.__version__, numpy_version=np.__version__,
    pandas_version=pd.__version__, python_version=platform.python_version(),
    features=FEATURES, official_attack_names=sorted(official_attack_names),
    development_attack_names=sorted(development_attack_names),
    fit_attack_names=sorted(fit_attack_names), primary_model=primary_name,
    preprocess_fit_and_validation_s=preprocess_fit_s)
(OUT / 'experiment_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2))
joblib.dump(dict(preprocessor=preprocessor, selected=selected, manifest=manifest),
    OUT / 'models' / 'project14_bundle.joblib', compress=3)

# Dự đoán thử sau khi đọc lại bundle: dữ liệu đầu vào chỉ cần 41 đặc trưng.
bundle = joblib.load(OUT / 'models' / 'project14_bundle.joblib')
new_rows = test_original[FEATURES].head(5).copy()
new_X = bundle['preprocessor'].transform(new_rows).astype(np.float32)
spec = bundle['selected'][bundle['manifest']['primary_model']]
new_score = anomaly_scores(spec, new_X)
display(pd.DataFrame({'score':new_score, 'prediction':np.where(
    new_score >= spec['threshold'], 'attack', 'normal')}))

archive = ROOT / 'project14_results.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in OUT.rglob('*'):
        if path.is_file():
            z.write(path, path.relative_to(ROOT))
print('Đã tạo:', archive, '| MiB:', round(archive.stat().st_size / 1024**2, 2))
try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download(str(archive))


## 14. Hoàn thiện bài tập lớn

**Bản đầu:** chạy hết notebook, giữ cả kết quả tốt lẫn kém. Không điền các con số kỳ vọng thay cho kết quả thực.

**Để bài sâu hơn:**
1. Lặp thí nghiệm bằng 3 seed, giữ test cố định, báo cáo trung bình và độ lệch chuẩn; không chọn seed tốt nhất.
2. Tăng ngân sách dữ liệu chung nếu tài nguyên cho phép. Nếu DBSCAN dùng ít mẫu hơn các mô hình khác, thêm thí nghiệm đó riêng và ghi rõ ngân sách.
3. So sánh có/không PCA hoặc các cách scale; cấu hình phải chọn bằng validation.
4. Có thể thêm thí nghiệm chỉ fit các mẫu normal. Phải ghi rõ đây là normal-only/novelty detection có dùng nhãn để chọn dữ liệu, tách khỏi thí nghiệm hỗn hợp.
5. Phân tích ít nhất một loại tấn công bị bỏ sót, một nhóm báo nhầm và tác động khi thay đổi ngưỡng. Không chỉnh mô hình theo test rồi vẫn báo đây là đánh giá độc lập; nếu thử tiếp theo thông tin test, cần tập kiểm thử mới cho kết luận mới.

**Khung báo cáo:** bài toán → dữ liệu → tiền xử lý → phương pháp và cách sử dụng nhãn → thiết kế thực nghiệm → kết quả tổng thể/novel → phân tích lỗi → hạn chế → kết luận.

**Hạn chế phải nêu:** NSL-KDD là benchmark lịch sử; các hàng là đặc trưng kết nối đã tổng hợp, không phải gói tin thô. Chạy thành công không đồng nghĩa có thể gắn trực tiếp mô hình vào mạng thật. Outlier không đồng nghĩa tấn công; cụm attack dày đặc có thể bị bỏ sót. Recall của tên tấn công ít mẫu có độ bất định cao.

## Tài liệu chính

- Dataset trong đề: https://www.kaggle.com/datasets/hassan06/nslkdd
- NSL-KDD, UNB/CIC: https://www.unb.ca/cic/datasets/nsl.html
- Định nghĩa đặc trưng KDD: https://kdd.ics.uci.edu/databases/kddcup99/task.html
- KMeans: https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html
- DBSCAN: https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html
- OneHotEncoder: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html
- LedoitWolf: https://scikit-learn.org/stable/modules/generated/sklearn.covariance.LedoitWolf.html
- RandomForestClassifier: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html
- Colab FAQ: https://research.google.com/colaboratory/faq.html

Khi nộp bài, trích dẫn NSL-KDD và bài báo Tavallaee et al. (2009), “A Detailed Analysis of the KDD CUP 99 Data Set”, theo yêu cầu nguồn dataset.
